# LightGBM: Histogram, GOSS, and Feature Bundling

Compare three LightGBM training configurations on the heart-disease dataset: the standard histogram-based GBDT baseline, Gradient-based One-Side Sampling (GOSS), and Exclusive Feature Bundling (EFB). This is an educational example only, not for medical diagnosis.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.base import clone

RANDOM_STATE = 42
print(f"LightGBM version: {lgb.__version__}")

## 1. Load and prepare the data

The source target `num` ranges from 0 to 4. We treat 0 as no disease and any value above 0 as disease. Identifier and source columns are excluded from model features.

In [ ]:
data_path_candidates = [
    Path("classification") / "heart_disease_uci.csv",
    Path("..") / "classification" / "heart_disease_uci.csv",
]
data_path = next((path for path in data_path_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find classification/heart_disease_uci.csv. "
        "Run this notebook from the repository root or the 'xg boost' folder."
    )

data = pd.read_csv(data_path)
if "num" not in data.columns:
    raise ValueError("Expected the target column 'num' in the heart-disease dataset.")

data = data.dropna(subset=["num"]).copy()
data["target"] = (data["num"] > 0).astype(int)
excluded_columns = {"id", "dataset", "num", "target"}
feature_columns = [column for column in data.columns if column not in excluded_columns]
if not feature_columns:
    raise ValueError("No usable feature columns were found.")

X = data[feature_columns]
y = data["target"]
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print(f"Loaded {len(data)} rows from {data_path}")
print(f"Features: {len(feature_columns)}")
display(y.value_counts().sort_index().rename(index={0: "No disease", 1: "Disease"}).to_frame("count"))

data.head()

In [ ]:
numeric_features

In [ ]:
categorical_features

In [ ]:
data.shape

## 2. Split the data

A stratified split keeps the class proportions similar in training and testing sets. Preprocessing is fitted only on training data.

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

In [ ]:
X_train

## 3. Preprocess and train three LightGBM configurations

Missing numeric values are filled with the training median, and missing categories with the most frequent value. Categories are one-hot encoded and the transformed matrix is kept sparse so EFB can bundle mutually exclusive one-hot features. LightGBM uses histogram-based split finding for all three models. The configurations below are: **1) histogram-based GBDT**, **2) GOSS** (`data_sample_strategy="goss"`), and **3) EFB feature bundling** (`enable_bundle=True`). EFB is disabled in the first two models and enabled in the third to make its effect explicit.

In [ ]:
numeric_preprocessing = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_preprocessing = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessing, numeric_features),
        ("categorical", categorical_preprocessing, categorical_features),
    ],
    sparse_threshold=1.0,
)

common_model_params = {
    "objective": "binary",
    "n_estimators": 100,
    "learning_rate": 0.05,
    "num_leaves": 15,
    "max_depth": 4,
    "max_bin": 63,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
    "verbosity": -1,
}

model_configs = {
    "Histogram-based GBDT": lgb.LGBMClassifier(
        **common_model_params,
        enable_bundle=False,
    ),
    "GOSS": lgb.LGBMClassifier(
        **common_model_params,
        data_sample_strategy="goss",
        top_rate=0.2,
        other_rate=0.1,
        enable_bundle=False,
    ),
    "Feature bundling (EFB)": lgb.LGBMClassifier(
        **common_model_params,
        enable_bundle=True,
    ),
}

classifiers = {}
for model_name, model in model_configs.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    classifiers[model_name] = pipeline
    print(f"Trained {model_name}")

# Keep a consistent reference model for the later single-model plots and example predictions.
classifier = classifiers["Histogram-based GBDT"]

In [ ]:
fitted_preprocessor = classifier.named_steps["preprocessor"]
matrix = fitted_preprocessor.transform(X_train)

# Convert sparse output to a regular array for display
if hasattr(matrix, "toarray"):
    matrix = matrix.toarray()

feature_matrix = pd.DataFrame(
    matrix,
    columns=fitted_preprocessor.get_feature_names_out(),
    index=X_train.index,
)

print("Matrix shape:", feature_matrix.shape)
display(feature_matrix.head())

In [ ]:
categorical_preprocessing

In [ ]:
classifier.named_steps["preprocessor"].feature_names_in_

## 4. Evaluate on the held-out test set

Compare the three configurations using the same held-out test split. Review several metrics rather than accuracy alone; precision and recall describe different types of classification errors. The histogram-based GBDT is used for the example plots below so model selection does not use test-set results.

In [ ]:
evaluation_rows = {}
test_predictions = {}
test_probabilities = {}
for model_name, pipeline in classifiers.items():
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    test_predictions[model_name] = predictions
    test_probabilities[model_name] = probabilities
    evaluation_rows[model_name] = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
    }

comparison = pd.DataFrame.from_dict(evaluation_rows, orient="index")
display(comparison.round(3))

baseline_name = "Histogram-based GBDT"
y_pred = test_predictions[baseline_name]
y_probability = test_probabilities[baseline_name]

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["No disease", "Disease"],
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["No disease", "Disease"], cmap="Blues"
)
plt.title("LightGBM histogram baseline: test-set confusion matrix")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_probability)
plt.title("LightGBM histogram baseline: test-set ROC curve")
plt.tight_layout()
plt.show()

## 5. Inspect feature importance and predict a sample

Feature importance shows how the fitted trees used features; it does not show cause and effect. The sample prediction below is taken from the test set.

In [ ]:
preprocessor_fitted = classifier.named_steps["preprocessor"]
model_fitted = classifier.named_steps["model"]
feature_names = preprocessor_fitted.get_feature_names_out()
feature_importance = (
    pd.Series(model_fitted.feature_importances_, index=feature_names)
    .sort_values(ascending=False)
    .head(15)
    .sort_values()
)
feature_importance.plot(kind="barh", figsize=(9, 6), title="Histogram baseline: top feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

sample_patient = X_test.iloc[[0]]
predicted_class = int(classifier.predict(sample_patient)[0])
disease_probability = float(classifier.predict_proba(sample_patient)[0, 1])
print(f"Predicted class: {predicted_class} (0 = no disease, 1 = disease)")
print(f"Estimated disease probability: {disease_probability:.1%}")
print(f"Actual class: {int(y_test.iloc[0])}")

## Recap

This notebook loads the repository's heart-disease data, splits it without leaking test data into preprocessing, and compares LightGBM's histogram-based GBDT, GOSS sampling, and EFB feature bundling. This example is for education only and must not be used to diagnose or make health decisions.

## Manual input prediction

Enter values for a new patient below. Press Enter at any prompt to use a suggested default from the training data. This example is for learning only, not medical diagnosis.

In [ ]:
print("Enter patient feature values. Press Enter to use the suggested default.\n")
manual_patient = {}

for column in feature_columns:
    values = X_train[column].dropna()
    if column in numeric_features:
        default_value = values.median()
        raw_value = input(f"{column} (number, default {default_value:g}): ").strip()
        manual_patient[column] = float(raw_value) if raw_value else float(default_value)
    else:
        default_value = values.mode().iloc[0]
        categories = values.astype(str).unique().tolist()
        print(f"  Options: {', '.join(categories[:10])}")
        raw_value = input(f"{column} (default {default_value}): ").strip()
        manual_patient[column] = raw_value if raw_value else default_value

manual_input = pd.DataFrame([manual_patient], columns=feature_columns)
manual_prediction = int(classifier.predict(manual_input)[0])
manual_probability = float(classifier.predict_proba(manual_input)[0, 1])

print("\nManual input prediction:")
print("Result: Disease detected" if manual_prediction == 1 else "Result: No disease detected")
print(f"Predicted class: {manual_prediction} (0 = no disease, 1 = disease)")
print(f"Estimated probability of disease: {manual_probability:.1%}")